In [2]:
import pyfastx
import gffutils
from src.create_dataset import createDataset
import os.path
import numpy as np

In [3]:
data_dir = '../Data/'
fasta_file_path = '../Data/hg38.fa'
gtf_file_path = '../Data/Homo_sapiens.GRCh38.87.gtf'


In [4]:
fasta = pyfastx.Fasta(fasta_file_path)

In [5]:
fname = data_dir+gtf_file_path.split('/')[-1][:-4]+'.db'
if not os.path.isfile(fname): 
    gffutils.create_db(gtf_file_path, fname, force=True, disable_infer_genes=True, disable_infer_transcripts=True)

gtf = gffutils.FeatureDB(fname)

In [6]:
def getJunctions(gtf,transcript_id):
    transcript = gtf[transcript_id.split('.')[0]]
    strand = transcript[6]
    exon_junctions = []
    tx_start = int(transcript[3])
    tx_end = int(transcript[4])
    exons = gtf.children(transcript, featuretype="exon")
    for exon in exons:
        exon_start = int(exon[3])
        exon_end = int(exon[4])
        exon_junctions.append((exon_start,exon_end))
        

    intron_junctions = []
    if strand=='+':
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    elif strand=='-':
        exon_junctions.reverse()
        intron_start = exon_junctions[0][1]
        for i,exon_junction in enumerate(exon_junctions[1:]):
            intron_end = exon_junction[0]
            intron_junctions.append((intron_start,intron_end))
            if i+1 != len(exon_junctions[1:]):
                intron_start = exon_junction[1]

    jn_start = [x[0] for x in intron_junctions]
    jn_end = [x[1] for x in intron_junctions]
    Y_type, Y_idx = [],[]
    if strand == '+':
        Y0 = -np.ones(tx_end-tx_start+1)
        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(c-tx_start)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(c-tx_start)

    elif strand == '-':
        Y0 = -np.ones(tx_end-tx_start+1)

        if len(jn_start) > 0:
            Y0 = np.zeros(tx_end-tx_start+1)
            for c in jn_end:
                if tx_start <= c <= tx_end:
                    Y_type.append(2)
                    Y_idx.append(tx_end-c)
            for c in jn_start:
                if tx_start <= c <= tx_end:
                    Y_type.append(1)
                    Y_idx.append(tx_end-c)

    return jn_start,jn_end,Y_type, Y_idx

In [9]:
genes = (gtf.features_of_type('transcript'))

for gene in genes:
    transcripts = gtf.children(gene, featuretype="transcript")
    for transcript in transcripts:
        transcript_id = transcript['transcript_id'][0]
        
        # getJunctions(gtf, transcript_id)


ENST00000456328
ENST00000456328
1	havana	transcript	11869	14409	.	+	.	gene_id "ENSG00000223972"; gene_version "5"; transcript_id "ENST00000456328"; transcript_version "2"; gene_name "DDX11L1"; gene_source "havana"; gene_biotype "transcribed_unprocessed_pseudogene"; havana_gene "OTTHUMG00000000961"; havana_gene_version "2"; transcript_name "DDX11L1-002"; transcript_source "havana"; transcript_biotype "processed_transcript"; havana_transcript "OTTHUMT00000362751"; havana_transcript_version "1"; tag "basic"; transcript_support_level "1";
ENST00000450305
ENST00000450305
1	havana	transcript	12010	13670	.	+	.	gene_id "ENSG00000223972"; gene_version "5"; transcript_id "ENST00000450305"; transcript_version "2"; gene_name "DDX11L1"; gene_source "havana"; gene_biotype "transcribed_unprocessed_pseudogene"; havana_gene "OTTHUMG00000000961"; havana_gene_version "2"; transcript_name "DDX11L1-001"; transcript_source "havana"; transcript_biotype "transcribed_unprocessed_pseudogene"; havana_transcript 

KeyboardInterrupt: 

In [5]:
print('Creating training data')
createDataset(gtf,fasta,'train',data_dir)

Creating training data


44697it [1:01:26, 12.12it/s]


KeyboardInterrupt: 

In [7]:
print('Creating test data')
createDataset(gtf,fasta,'test',data_dir)

Creating test data


58051it [44:09, 21.91it/s]   
